# 01v2 - A0b: verify DICOM slice ordering

**Plan item A0b** (2026-08-25 reorientation, see README.md /
[[project-rsna-phase-status]]): the now-obsolete `06`/`06b` sort each
series' slices by bare `ds.SliceLocation` only. A same-competition
forum post (stevenleehans, discussion 735304) measured that filename/
SOP-instance order matches true anatomical slice order only ~5% of the
time on this corpus, and that the reliable method is projecting
`ImagePositionPatient` onto the normal of `ImageOrientationPatient`
(`cross(row_dir, col_dir)`), with a `SliceLocation` -> `InstanceNumber`
-> filename fallback chain -- which resolved 100% of series for them.

This notebook checks, on the 58 real gold studies, whether that
geometric method ever disagrees with the current bare-`SliceLocation`
sort -- and if so, how often and how badly. Metadata-only (DICOM headers
via `stop_before_pixels=True`, no pixel decode) -- fast, CPU-only, no
GPU needed.

**Effort estimate (plan artifact):** ~1 hour, read-only. **Gates:** how
we interpret the unexplained ~10/58 residual noted in the now-obsolete
`06b`, and the slice-position design work in A2.

## Cell 1 - Imports, mount, select one sagittal series per gold study

Series-selection logic (`count_slices`, `select_sagittal_series`)
reused verbatim from the now-obsolete `06b` -- it's unrelated to the
ordering question this notebook checks, and was already validated
there.

In [ ]:
from pathlib import Path

import pandas as pd

RAW_DIR = Path("/kaggle/input/competitions/rsna-knee-abnormality-detection")
assert RAW_DIR.exists(), f"Competition data not found at {RAW_DIR}"

OFFICIAL_LABEL_COLUMNS = {
    "acl_injury": "ACL", "mcl_injury": "MCL",
    "medial_meniscus_tear": "Medial Meniscus", "lateral_meniscus_tear": "Lateral Meniscus",
    "oa_medial_compartment": "Medial OA", "oa_lateral_compartment": "Lateral OA",
    "oa_patellofemoral_compartment": "PF OA", "effusion": "Effusion",
    "synovitis": "Synovitis", "bakers_cyst": "Baker's",
    "bone_contusion": "Contusion", "fracture": "Fracture",
}
LABEL_COLS = list(OFFICIAL_LABEL_COLUMNS.values())

train = pd.read_csv(RAW_DIR / "train.csv")
train_series = pd.read_csv(RAW_DIR / "train_series.csv")

gold_mask = train[LABEL_COLS].notna().all(axis=1)
gold = train.loc[gold_mask, ["StudyInstanceUID"]].reset_index(drop=True)
assert len(gold) == 58, f"Expected 58 gold studies, found {len(gold)}"
print(f"Gold studies: {len(gold)}")

gold_sag = train_series[
    train_series["StudyInstanceUID"].isin(gold["StudyInstanceUID"])
    & (train_series["Anatomical_Plane"].str.lower() == "sagittal")
]
print(f"Sagittal series among gold: {len(gold_sag)} rows, covering {gold_sag['StudyInstanceUID'].nunique()} / 58 studies")


def count_slices(study_id, series_id, series_subdir):
    d = RAW_DIR / series_subdir / study_id / series_id
    return len(list(d.glob("*.dcm")))


def select_sagittal_series(series_df, slice_counts):
    sagittal = series_df.copy()
    sagittal["n_slices"] = sagittal["SeriesInstanceUID"].map(slice_counts).fillna(0).astype(int)

    def _pick(group):
        fluid_sensitive = group[group["Fluid_Sensitive"] == 1]
        pool = fluid_sensitive if len(fluid_sensitive) > 0 else group
        pool = pool.sort_values(["n_slices", "SeriesInstanceUID"], ascending=[False, True])
        return pool.iloc[0]

    return sagittal.groupby("StudyInstanceUID").apply(_pick, include_groups=False)


_slice_counts = {}
for _, row in gold_sag.iterrows():
    key = row["SeriesInstanceUID"]
    _slice_counts[key] = count_slices(row["StudyInstanceUID"], key, "train_series")

selected = select_sagittal_series(gold_sag, _slice_counts)
print(f"Series selected: {len(selected)} / 58 gold studies")
print(selected[["SeriesInstanceUID", "n_slices"]].describe())

## Cell 2 - Compute both orderings and compare

`geometric_slice_order`: primary key is `ImagePositionPatient` projected
onto the normal of `ImageOrientationPatient` (`cross(row_dir, col_dir)`),
falling back to `SliceLocation`, then `InstanceNumber`, then filename
order when the geometric tags are missing or degenerate (zero-norm
normal). Source: stevenleehans, this competition's discussion 735304.

`current_slice_order`: the now-obsolete `06`/`06b` method -- bare
`SliceLocation` sort, no fallback.

For each of the 58 gold series: read all headers (`stop_before_pixels=
True`), compute both orders, and check whether they agree -- and if
not, whether it's a pure reversal (same set, opposite direction) or a
real permutation (which a downstream fixed-direction laterality flip
couldn't fix).

In [ ]:
import numpy as np
import pydicom


def geometric_slice_order(files):
    """Sort DICOM slices by physical position along the series normal.

    Primary key: ImagePositionPatient projected onto the normal of
    ImageOrientationPatient (cross(row_dir, col_dir)). Falls back to
    SliceLocation, then InstanceNumber, then filename order when the
    geometric tags are missing or degenerate (zero-norm normal).
    Source: stevenleehans, this competition's discussion 735304.
    """
    records = []
    for idx, f in enumerate(files):
        ds = pydicom.dcmread(f, stop_before_pixels=True)
        key, method = None, None

        iop = getattr(ds, "ImageOrientationPatient", None)
        ipp = getattr(ds, "ImagePositionPatient", None)
        if iop is not None and ipp is not None and len(iop) == 6 and len(ipp) == 3:
            row_dir = np.array(iop[0:3], dtype=float)
            col_dir = np.array(iop[3:6], dtype=float)
            normal = np.cross(row_dir, col_dir)
            if np.linalg.norm(normal) > 1e-6:
                key = float(np.dot(np.array(ipp, dtype=float), normal))
                method = "geometric"

        if key is None:
            sl = getattr(ds, "SliceLocation", None)
            if sl is not None:
                key, method = float(sl), "slice_location"
        if key is None:
            inum = getattr(ds, "InstanceNumber", None)
            if inum is not None:
                key, method = float(inum), "instance_number"
        if key is None:
            key, method = float(idx), "filename"

        records.append((key, method, f))

    records.sort(key=lambda r: r[0])
    return [f for _, _, f in records], [m for _, m, _ in records]


def current_slice_order(files):
    """Current (now-obsolete 06/06b) method: bare SliceLocation sort, no fallback."""
    keyed = [(float(pydicom.dcmread(f, stop_before_pixels=True).SliceLocation), f) for f in files]
    keyed.sort(key=lambda r: r[0])
    return [f for _, f in keyed]


order_rows = []
for study_id in selected.index:
    series_id = selected.loc[study_id, "SeriesInstanceUID"]
    d = RAW_DIR / "train_series" / study_id / series_id
    files = sorted(d.glob("*.dcm"))

    geo_files, geo_methods = geometric_slice_order(files)
    cur_files = current_slice_order(files)

    order_rows.append(dict(
        study_id=study_id,
        n_slices=len(files),
        methods_used=",".join(sorted(set(geo_methods))),
        orders_match=(geo_files == cur_files),
        is_pure_reversal=(geo_files == list(reversed(cur_files))),
    ))

order_df = pd.DataFrame(order_rows)
print(f"Studies where geometric order == current SliceLocation order: {order_df['orders_match'].sum()} / {len(order_df)}")
print()
print("Geometric-key method actually used across the corpus (should be ~all 'geometric' if IOP/IPP tags are populated):")
print(order_df["methods_used"].value_counts())
print()

mismatches = order_df[~order_df["orders_match"]]
if len(mismatches):
    print(f"Mismatches ({len(mismatches)}):")
    print(mismatches[["study_id", "n_slices", "is_pure_reversal"]].to_string())
else:
    print("No mismatches -- SliceLocation-only sort agrees with the geometric method on every gold series.")

## Cell 3 - Why would they agree or disagree? (gantry tilt + tag reliability)

Bare `SliceLocation` and the geometric projection are mathematically
identical when a series has zero gantry tilt (`ImageOrientationPatient`
axis-aligned) and `SliceLocation` is populated without ties. This cell
measures both conditions directly, so Cell 2's result (match/mismatch)
has an explanation instead of being a bare yes/no:

- **Tilt**: degrees between the series normal and the nearest standard
  axis. Near 0 explains agreement; non-trivial tilt is where the two
  methods could genuinely diverge.
- **Tag reliability**: whether `ImageOrientationPatient`/
  `ImagePositionPatient` are present at all, and whether `SliceLocation`
  has duplicate values (ties make sort order depend on filesystem
  iteration order, not anatomy).

In [ ]:
tilt_rows = []
for study_id in selected.index:
    series_id = selected.loc[study_id, "SeriesInstanceUID"]
    d = RAW_DIR / "train_series" / study_id / series_id
    files = sorted(d.glob("*.dcm"))
    ds0 = pydicom.dcmread(files[0], stop_before_pixels=True)

    iop = getattr(ds0, "ImageOrientationPatient", None)
    ipp0 = getattr(ds0, "ImagePositionPatient", None)
    tilt_deg = None
    if iop is not None and len(iop) == 6:
        row_dir = np.array(iop[0:3], dtype=float)
        col_dir = np.array(iop[3:6], dtype=float)
        normal = np.cross(row_dir, col_dir)
        norm_len = np.linalg.norm(normal)
        if norm_len > 1e-6:
            normal = normal / norm_len
            axis_alignment = max(abs(normal[0]), abs(normal[1]), abs(normal[2]))
            tilt_deg = float(np.degrees(np.arccos(min(1.0, axis_alignment))))

    slice_locations = [float(pydicom.dcmread(f, stop_before_pixels=True).SliceLocation) for f in files]
    n_unique = len(set(slice_locations))

    tilt_rows.append(dict(
        study_id=study_id,
        has_iop_ipp=(iop is not None and ipp0 is not None),
        tilt_deg=tilt_deg,
        n_slices=len(files),
        n_unique_slice_locations=n_unique,
        has_duplicate_slice_locations=(n_unique < len(files)),
    ))

tilt_df = pd.DataFrame(tilt_rows)
print("Series missing ImageOrientationPatient/ImagePositionPatient entirely:")
print(f"{(~tilt_df['has_iop_ipp']).sum()} / {len(tilt_df)}")
print()
print("Gantry tilt (degrees off the nearest standard axis) distribution:")
print(tilt_df["tilt_deg"].describe())
print()
print("Series with duplicate SliceLocation values (ties -- ordering becomes ambiguous):")
print(f"{tilt_df['has_duplicate_slice_locations'].sum()} / {len(tilt_df)}")
if tilt_df["has_duplicate_slice_locations"].any():
    print(tilt_df[tilt_df["has_duplicate_slice_locations"]][["study_id", "n_slices", "n_unique_slice_locations"]])
print()

merged = order_df.merge(tilt_df[["study_id", "tilt_deg", "has_duplicate_slice_locations"]], on="study_id")
print("Cross-reference: tilt_deg for order-matched vs order-mismatched studies")
print(merged.groupby("orders_match")["tilt_deg"].describe())

## Interpretation and decision

Read Cell 2 and Cell 3's output together:

- **0 mismatches in Cell 2, tilt ~0 deg across the corpus in Cell 3:**
  bare `SliceLocation` happens to be safe *on this specific gold
  sample* because these series are all near axis-aligned -- not because
  the method is inherently robust. The slice-ordering bug this cell was
  built to catch is a **near-miss, not a false alarm**: a series with
  real gantry tilt (routine on other scanners/protocols even if absent
  here) would silently break bare `SliceLocation` sort and there would
  be no signal in the pipeline to catch it. Adopt
  `geometric_slice_order` anyway before A3 -- it's strictly more
  correct, degrades gracefully via the fallback chain, and costs
  nothing extra to compute. Log this as a negative result for "is there
  an active bug in the 58 gold studies" but a positive result for "is
  the fix worth making" per [[structuring-ml-projects]] step 7.

- **Any mismatches in Cell 2, concentrated in the higher-`tilt_deg`
  rows of Cell 3's cross-reference:** confirms the geometric method is
  not just theoretical insurance -- it changes real ordering on this
  corpus. `geometric_slice_order` must replace the bare-`SliceLocation`
  sort in A3, and the mismatched study IDs are worth checking against
  the now-obsolete `06b`'s ~10/58 residual list for overlap (that
  residual was never explained -- this could be why).

- **Mismatches uncorrelated with tilt, but concentrated in the
  `has_duplicate_slice_locations` rows:** a tag-reliability issue, not
  a geometry issue -- ties in `SliceLocation` make plain sort ambiguous
  regardless of tilt. Same fix (adopt `geometric_slice_order`, its
  fallback chain doesn't help with duplicate *geometric* keys either,
  so duplicates specifically may need a secondary tie-break on
  `InstanceNumber` -- flag this back if it comes up).

**Either way, next step is the same:** report the actual printed
tables back, then -- once reviewed -- graduate `geometric_slice_order`
into `src/data.py` with a unit test covering the geometric-key path,
each fallback step, and the tie-break case, per this project's
notebook-to-`src/` graduation rule. Don't write to `src/` before that
review, even if the result looks clearly like "no bug."